In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import roc_auc_score, brier_score_loss
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline

# Load cleaned data
df = pd.read_csv("cleveland_cleaned.csv")

X = df.drop(columns=["num", "target"])
y = df["target"].values

# Categorical and numeric feature groups
cat_cols = ["sex", "cp", "fbs", "restecg", "exang", "slope", "ca", "thal"]
num_cols = [c for c in X.columns if c not in cat_cols]

preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ]
)

# Base classifiers
log_reg = LogisticRegression(max_iter=1000, solver="liblinear")
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=4,
    min_samples_leaf=5,
    random_state=42,
)

outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def nested_probs(base_clf, method=None):
    """Return out-of-fold probabilities under nested CV."""
    probs = np.zeros_like(y, dtype=float)

    for train_idx, test_idx in outer_cv.split(X, y):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train = y[train_idx]

        if method is None:
            model = Pipeline(
                steps=[
                    ("prep", preprocess),
                    ("clf", base_clf),
                ]
            )
        else:
            model = Pipeline(
                steps=[
                    ("prep", preprocess),
                    (
                        "calib",
                        CalibratedClassifierCV(
                            base_estimator=base_clf,
                            cv=3,
                            method=method,
                        ),
                    ),
                ]
            )

        model.fit(X_train, y_train)
        probs[test_idx] = model.predict_proba(X_test)[:, 1]

    return probs

# Compute probabilities for all models
p_base = np.full_like(y, fill_value=y.mean(), dtype=float)

lr_raw = nested_probs(log_reg, method=None)
lr_platt = nested_probs(log_reg, method="sigmoid")
rf_raw = nested_probs(rf, method=None)
rf_iso = nested_probs(rf, method="isotonic")

results = {
    "Baseline": p_base,
    "LogReg (raw)": lr_raw,
    "LogReg + Platt": lr_platt,
    "RF (raw)": rf_raw,
    "RF + Isotonic": rf_iso,
}

print(f"{'Model':<18}  AUC   Brier")
for name, p in results.items():
    auc = roc_auc_score(y, p)
    brier = brier_score_loss(y, p)
    print(f"{name:<18}  {auc:5.3f}  {brier:6.3f}")

# Reliability diagram
plt.figure(figsize=(7, 6))
plt.plot([0, 1], [0, 1], "k:", label="Perfect")

for name, p in results.items():
    frac_pos, mean_pred = calibration_curve(y, p, n_bins=10)

    # solid line for calibrated models, dashed for raw / baseline
    style = "-" if "+" in name else "--"
    plt.plot(mean_pred, frac_pos, style, marker=".", label=name)

plt.xlabel("Mean predicted probability")
plt.ylabel("Fraction of positives")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()
